## Rendimientos simples y logarítmicos

In [26]:
#Importando librerías
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_datareader.data as web
from datetime import datetime,timedelta
from dateutil import relativedelta

In [27]:
#Declaración de variables

now = datetime.now()
# primer dia del mes reciente
f_day_of_month = now.replace(day=1)
#Declarar el código bursátil de la empresa Microsoft
stock_code='MSFT'

In [28]:
# Descargar los precios cerrados con ajuste desde el primer día del mes hasta hoy
df = yf.download(stock_code,
                 start=f_day_of_month.strftime('%Y-%m-%d'),
                 end=now.strftime('%Y-%m-%d'),
                 progress=False,
                 auto_adjust=False
                )

# Seleccione la columna MultiIndex específica para 'Adj Close' y el ticker 'MSFT', 
# luego conviértalo en un DataFrame con una sola columna llamada 'adj_close'.
df = df[('Adj Close',stock_code)].to_frame(name='adj_close')


In [29]:
# Calcullar los rendimientos simples y logarítmicos
df['simple_rtn']= df.adj_close.pct_change()
df['log_rtn']= np.log(df.adj_close/df.adj_close.shift(1))
df

,adj_close,simple_rtn,log_rtn
Date,,,
2025-11-03,516.064148,NaN,NaN
2025-11-04,513.369202,-0.005222,-0.005236
2025-11-05,506.212555,-0.013941,-0.014039
2025-11-06,496.171356,-0.019836,-0.020035
2025-11-07,495.891876,-0.000563,-0.000563
2025-11-10,505.054718,0.018477,0.018309
2025-11-11,507.729706,0.005296,0.005282
2025-11-12,510.185150,0.004836,0.004824
2025-11-13,502.349792,-0.015358,-0.015477



## Retornos con tasa de inflación

In [34]:
#Declarar variables de fecha
from datetime import datetime
from dateutil.relativedelta import relativedelta
start_date = datetime.now().replace(month=1).replace(day=1)
end_date =  (((datetime.now()- relativedelta(months=1))).replace(day=1))- relativedelta(days=1)
start_date_2= (start_date - relativedelta(months=1)).replace(day=1)

#Descarga los precios de cierre ajustados desde la fecha inicial  a la fecha final
df = yf.download(stock_code,
                 start=start_date.strftime('%Y-%m-%d'),
                 end=end_date.strftime('%Y-%m-%d'),
                 progress=False,
                 auto_adjust=False
                )


# Seleccione la columna MultiIndex específica para 'Adj Close' y el ticker 'MSFT',
# luego conviértalo en un DataFrame con una sola columna llamada 'adj_close'.
df = df[('Adj Close',stock_code)].to_frame(name='adj_close')

#Crear un dataframe con la fusión entre las fechas (izquierda) y los precios de cierre (derecha)
df_all_dates= pd.DataFrame(index=pd.date_range(start=start_date_2.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d')))
df=df_all_dates.join( df, how ='left') \
   .ffill() \
   .asfreq('ME')

# Descargar los datos del IPC -tasa de inflación- EE.UU.
df_cpi = web.DataReader('CPIAUCNS', 'fred', start=start_date_2.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
df_cpi.rename(columns={'CPIAUCNS': 'cpi'}, inplace=True)


# Alinear el índice df_cpi al final del mes para que coincida con la frecuencia de df
df_cpi_aligned = df_cpi.copy()
df_cpi_aligned.index = df_cpi_aligned.index + pd.offsets.MonthEnd(0)

# Remuestrear según la frecuencia de fin de mes y completar hacia adelante los valores faltantes (si los hay)
df_cpi_aligned = df_cpi_aligned.resample('ME').ffill()


#Fusión entre la tasa de inflación y los precios
df_merged = pd.merge(df, df_cpi_aligned, left_index=True, right_index=True, how='left')

# Calcular los rendimientos simples y la tasa de inflación
df_merged ['simple_rtn']= df.adj_close.pct_change()
df_merged['inflation_rate']=df_merged.cpi.pct_change()

#Calcular los rendimientos ajustados a la inflación
df_merged['real_rtn']=(df_merged.simple_rtn + 1) / (df_merged.inflation_rate +1) - 1


In [33]:
df_merged

,adj_close,cpi,simple_rtn,inflation_rate,real_rtn
2024-12-31,NaN,315.605,NaN,NaN,NaN
2025-01-31,412.020569,317.671,NaN,0.006546,NaN
2025-02-28,394.873108,319.082,-0.041618,0.004442,-0.045856
2025-03-31,373.388306,319.799,-0.054409,0.002247,-0.056529
2025-04-30,393.152344,320.795,0.052932,0.003114,0.049662
2025-05-31,458.745819,321.465,0.166840,0.002089,0.164408
2025-06-30,495.665955,322.561,0.080481,0.003409,0.076809
2025-07-31,531.629395,323.048,0.072556,0.001510,0.070939
2025-08-31,505.743439,323.976,-0.048692,0.002873,-0.051417
2025-09-30,513.638611,324.800,0.015611,0.002543,0.013034
